# 04 – Availability and rental model

## Question
- **Question:** Is Airbnb in these cities still a short-stay (tourist) product, or has a part of it turned into mid- and long-term rental (30+ nights)? How does the rental model relate to how available a listing is during the year?
- **Hypothesis (before looking at the data):** long-term listings (`minimum_nights >= 30`) are more common in Los Angeles than in Barcelona.
- **Variables:** `minimum_nights`, `availability_365`, `room_type`, `neighbourhood_cleansed`.
- **Methods:** missing-value check, frequency table of `minimum_nights`, share of 30+ listings per city, histograms and boxplots of `availability_365`, statistical tests (chi-square, Mann-Whitney), choropleth map of the 30+ share per neighbourhood, replication on pairs 2 and 3 with the same code.

## 1. Load the data

Load only the columns this study needs (19 of 90) for the cities of pair 1 (Barcelona, Los Angeles), the discovery pair. The replication later runs the same cells for pairs 2 and 3. A readable `city_name` column is added for plots, and the last line shows the number of listings per city.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.cities import CITIES, cities_in_pair
from src.load import load_all
from src.maps import add_choropleth, add_heatmap, base_map, save_map

FIG_DIR = Path("../figures")  # PNGs for the report
FIG_DIR.mkdir(exist_ok=True)

cols = [
    "id", "host_id", "room_type", "neighbourhood_cleansed", "neighbourhood_group_cleansed",
    "latitude", "longitude", "price",
    "minimum_nights", "maximum_nights", "minimum_minimum_nights", "maximum_minimum_nights",
    "has_availability", "availability_30", "availability_90", "availability_365",
    "instant_bookable", "number_of_reviews_ltm", "estimated_occupancy_l365d",
    "license", "calculated_host_listings_count",
]

# Discovery on pair 1 first; replication later runs the same cells with all cities
df = load_all(cities_in_pair(1), usecols=cols)
df["city_name"] = df["city"].map({k: c.name for k, c in CITIES.items()})
df.groupby("city_name").size()

## 2. Missing values

Before any analysis, check which of the loaded columns have missing values, as a percentage per city. This tells us which variables can be used at all and whether a missing share differs between cities (which would itself need an explanation).

**Result:** `instant_bookable` is empty (100% missing) in both cities, and in all six cities of the project, so it is dropped from the study. The core variables of this study (`minimum_nights`, `availability_365`, `room_type`) are complete.

In [ ]:
# Share of missing values (%) per column and city; only columns with any missing values are shown
missing = df[cols].isna().groupby(df["city_name"]).mean().mul(100).round(1).T
missing[missing.max(axis=1) > 0]

## 3. Are listings without a price random?

`price` is missing for 12.7% of listings. Dropping these rows is only safe if they are a random subset. To check, compare listings with and without a price on availability and on recent reviews (a demand proxy).

**Result:** listings without a price have a median `availability_365` of 0, and about 72% of them have no free night at all, against 0% of listings with a price. They also have a median of 0 reviews in the last 12 months. The missing price is therefore not random: it marks listings that cannot be booked at the moment. Our interpretation is that these are inactive ("ghost") listings that are still visible on the platform. **Rows without a price are not dropped:** price is not a variable in this study, and dropping them would remove exactly the group that matters for an availability study (selection bias). Analyses that need the price drop missing values locally.

In [ ]:
# Compare listings with and without a price: availability and recent demand
df["price_missing"] = df["price"].isna()
df.groupby(["city_name", "price_missing"]).agg(
    listings=("id", "size"),
    median_availability_365=("availability_365", "median"),
    pct_availability_0=("availability_365", lambda s: (s == 0).mean() * 100),
    median_reviews_ltm=("number_of_reviews_ltm", "median"),
).round(1)

## 4. Inactive ("ghost") listings

A listing with `availability_365 == 0` is either booked out for the whole year or blocked by the host. Reviews separate the two cases: a listing that is booked out should keep receiving reviews. We define an **inactive listing** as one with zero availability **and** no review in the last 12 months, and keep it in the data with a flag (`is_inactive`) instead of dropping it. Later analyses of availability can then be run with and without these listings.

**Result:** about 9% of listings in both cities have zero availability, and most of them (86% in Barcelona, 82% in Los Angeles) had no review in the last 12 months. Inactive listings make up 7.8% of Barcelona and 7.6% of Los Angeles. The few zero-availability listings with recent reviews (196 and 728) may be booked out or blocked only recently, so they are not flagged.

In [ ]:
# Flag "ghost" listings: no free night in the next 365 days and no review in the last 12 months
df["is_inactive"] = (df["availability_365"] == 0) & (df["number_of_reviews_ltm"] == 0)

# Of the listings with zero availability, how many still got reviews (booked out) and how many did not (ghost)?
zero_avail = df[df["availability_365"] == 0]
print(pd.crosstab(zero_avail["city_name"], zero_avail["number_of_reviews_ltm"] > 0)
        .rename(columns={False: "no reviews in 12 months", True: "has reviews"}))

# Share of inactive listings (%) per city
df.groupby("city_name")["is_inactive"].mean().mul(100).round(1)

## 5. Minimum stay: the most common values

`minimum_nights` is the shortest stay a host accepts. It is the clearest signal of the rental model a host has chosen: 1–2 nights is a tourist product, 30+ nights is closer to a monthly rental. The table lists the 8 most common values per city (as % of the city's listings). The chart shows the whole distribution up to 40 nights, one panel per city.

**Result:** the distribution is bimodal in both cities: one peak at 1–2 nights and a second peak at about one month, with almost nothing between 7 and 29 nights. The position of the second peak differs. In Barcelona it sits at **31 and 32 nights** (17.3% each), while exactly 30 nights is only 1.0%. In Los Angeles it sits at **30 nights** (35.7%, the most common value of all), followed by 31 nights (9.6%).

**Interpretation:** the second peak sits at each city's legal boundary between a tourist rental and an ordinary rental.
- **Barcelona:** in Catalonia, a tourist-use dwelling (*habitatge d'ús turístic*, HUT) is a dwelling let for stays of **31 days or less**, and it needs a tourist licence (HUTB number). Barcelona has issued no new licences since 2014 and plans to phase out the existing ones by 2028. A stay longer than 31 days is a seasonal rental (*lloguer de temporada*) under the national Urban Leases Law (LAU) and needs no tourist licence.
- **Los Angeles:** the Home-Sharing Ordinance (Ordinance 185931, in force since 2019) defines a short-term rental as transient use of **30 consecutive days or less**. Short-term renting requires a city registration, is allowed only in the host's primary residence, and is capped at 120 days a year. A host who rents a second home, or does not want to register, can only offer longer stays.
- In both cities the peak is split over two values (31/32 and 30/31). This is consistent with hosts being unsure whether the limit counts nights or calendar days and adding one night to stay safely outside it. This part is a hypothesis: the data shows where hosts put the minimum, not why. Section 7 tests the Barcelona explanation against the licence field.

Sources (accessed 2026-10-25): Generalitat de Catalunya, *Declaració responsable d'habitatges d'ús turístic* (https://www.accio.gencat.cat/en/serveis/convocatories-dajuts/llistat-ajuts/tramits/Declaracio-responsable-dhabitatges-dus-turistic-a-lajuntament); City of Los Angeles, Home-Sharing Ordinance (https://planning.lacity.gov/ordinances/docs/HomeSharing/adopted/Final%20Ordinance.pdf) and Council File 14-1635-S7 (https://cityclerk.lacity.org/onlinedocs/2014/14-1635-S7_misc_08-27-2020.pdf); licence freeze and 2028 phase-out in Barcelona as summarised by secondary sources, e.g. https://www.hostaway.com/blog/airbnb-rules-barcelona/.

In [ ]:
# The 8 most common minimum_nights values per city, as % of the city's listings
(
    df.groupby("city_name")["minimum_nights"]
    .value_counts(normalize=True)
    .mul(100).round(1)
    .groupby("city_name").head(8)
    .rename("pct_of_listings")
    .reset_index()
    .astype({"minimum_nights": "Int64"})  # stored as float because of missing values; nights are whole numbers
)

In [ ]:
def plot_min_nights(data: pd.DataFrame, filename: str) -> None:
    """Distribution of minimum_nights from 1 to 40 nights, one panel per city (% of all listings of the city)."""
    city_names = data["city_name"].unique()
    ncols = min(3, len(city_names))
    nrows = -(-len(city_names) // ncols)  # ceiling division: 2 cities -> 1 row, 6 cities -> 2 rows
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.5 * ncols + 1, 3.6 * nrows), sharey=True, squeeze=False)
    for ax, city in zip(axes.flat, city_names):
        nights = data.loc[data["city_name"] == city, "minimum_nights"]
        pct = nights[nights <= 40].value_counts().sort_index() / len(nights) * 100
        ax.bar(pct.index, pct.values, color="#2a78d6", width=0.8)
        ax.set_title(city)
        ax.set_xlabel("Minimum nights")
        ax.set_xticks([1, 7, 14, 21, 30, 40])
        ax.grid(axis="y", alpha=0.3)
        ax.set_axisbelow(True)
        ax.spines[["top", "right"]].set_visible(False)
    for ax in axes[:, 0]:
        ax.set_ylabel("% of listings")
    for ax in axes.flat[len(city_names):]:
        ax.set_visible(False)  # hide empty panels when the grid has more panels than cities
    fig.suptitle("Minimum stay required by hosts (values above 40 nights not shown)")
    plt.tight_layout()
    plt.savefig(FIG_DIR / filename, dpi=300, bbox_inches="tight")
    plt.show()


plot_min_nights(df, "study04_min_nights_distribution.png")

## 6. Rental model per city (prediction check)

Group listings into three rental models by `minimum_nights`: **short stay** (1–6 nights, a classic tourist product), **weekly** (7–29) and **monthly+** (30+, closer to a mid- or long-term rental). The cut-offs are numbers of nights, not money, so the same thresholds are valid in every city. The 12 listings with a missing `minimum_nights` get no group.

**Result:** monthly+ listings make up **37.1% in Barcelona and 46.8% in Los Angeles** (short stay: 60.4% vs 49.7%; weekly: 2.5% vs 3.5%). The direction of the team's prediction holds: there are more 30+ listings in Los Angeles. The size is the surprise: in both cities more than a third of all listings cannot be booked as a short tourist stay at all.

**Interpretation:** a minimum of one month makes these listings **mid-term (seasonal) rentals** rather than long-term housing. Likely guests are students, interns, remote workers or people relocating to the city, but the data contains no information about guests, so this is a hypothesis. Such listings compete less with hotels and more with the ordinary rental market.

In [ ]:
# Group listings into three rental models by the minimum stay the host accepts
bins = [0, 6, 29, np.inf]  # (0, 6] = 1-6 nights, (6, 29] = 7-29 nights, (29, inf) = 30+ nights
labels = ["Short stay (1-6 nights)", "Weekly (7-29)", "Monthly+ (30+)"]
df["rental_model"] = pd.cut(df["minimum_nights"], bins=bins, labels=labels)
df["rental_model"].value_counts(dropna=False)

In [ ]:
def plot_rental_model_share(data: pd.DataFrame, filename: str) -> pd.DataFrame:
    """Share of listings per rental model and city (%) as a stacked bar chart. Returns the table."""
    share = pd.crosstab(data["city_name"], data["rental_model"], normalize="index").mul(100).round(1)

    colors = ["#86b6ef", "#2a78d6", "#104281"]  # one blue hue: lighter = shorter stays, darker = longer stays
    ax = share.plot(kind="barh", stacked=True, color=colors, figsize=(9, 1.6 + 0.6 * len(share)),
                    width=0.6, edgecolor="white", linewidth=2)
    for container, text_color in zip(ax.containers, ["black", "white", "white"]):
        labels = [f"{v:.1f}%" if v >= 5 else "" for v in container.datavalues]  # too narrow to label below 5%
        ax.bar_label(container, labels=labels, label_type="center", color=text_color, fontsize=10)
    ax.invert_yaxis()  # first city on top
    ax.set_xlim(0, 100)
    ax.set_xlabel("% of listings")
    ax.set_ylabel("")
    ax.set_title("Rental model by city (from minimum_nights)")
    ax_height = ax.get_window_extent().height / ax.figure.dpi  # in inches, to keep the legend at a fixed gap below
    ax.legend(title="", ncol=3, loc="upper center", bbox_to_anchor=(0.5, -0.75 / ax_height), frameon=False)
    ax.spines[["top", "right"]].set_visible(False)
    plt.savefig(FIG_DIR / filename, dpi=300, bbox_inches="tight")
    plt.show()
    return share


plot_rental_model_share(df, "study04_rental_model_share.png")

## 7. Barcelona: licence status by rental model

If the 31–32-night peak is caused by the tourist-licence rules, monthly+ listings in Barcelona should **not** carry a tourist licence. In Barcelona the `license` field holds the regional number (`HUTB-...` for a tourist flat), the national registration number, or an explicit exemption such as `Exempt - seasonal rental`. The field has a different format in every city, so this check is done for Barcelona only and is not part of the replication.

**Result:** the licence field splits cleanly by rental model. **77.3% of short-stay listings carry a tourist licence (HUTB), against only 1.4% of monthly+ listings.** In turn, 52.2% of monthly+ listings are explicitly marked *exempt: seasonal rental*, a label that never appears among short-stay listings. Another 30.5% of monthly+ listings have no licence entry at all, which is legal for a stay longer than 31 days. This supports the interpretation in section 5: in Barcelona the monthly+ market is the part of Airbnb that operates outside the tourist-licence system.

In [ ]:
# Barcelona only: what the licence field says, per rental model (% of listings in each rental model)
bcn = df[df["city"] == "barcelona"]
licence = bcn["license"].fillna("")

licence_status = pd.Series("other (hotel, hostel, ...)", index=bcn.index, name="licence_status")
licence_status[licence == ""] = "missing"
licence_status[licence.str.contains("HUTB")] = "tourist licence (HUTB)"
licence_status[licence.str.contains("seasonal rental", case=False)] = "exempt: seasonal rental"

pd.crosstab(licence_status, bcn["rental_model"], normalize="columns").mul(100).round(1)

## 8. Are monthly+ listings really rented for a month?

A host could set a 30+ minimum only on paper and still rent out short stays. Reviews give an upper bound: a listing with a minimum of N nights can host at most `365 // N` stays a year, and each stay produces at most one review. A monthly+ listing with more reviews in the last 12 months than that is **implausible** as a pure monthly rental.

**Result:** only **0.5% (Barcelona) and 1.1% (Los Angeles)** of monthly+ listings exceed the upper bound. The median monthly+ listing got 0 reviews in the last 12 months, against 13 and 9 for short-stay listings. So the 30+ minimum is in almost all cases real, not a cover for short stays. **Limitation:** `minimum_nights` is the value at scrape time. A host who switched from short stays to 30+ during the year also counts as implausible, so 0.5–1.1% is an upper bound for disguised short stays. Hosts who rent short stays off-platform cannot be detected at all.

In [ ]:
# A stay of at least N nights allows at most 365 // N stays a year, and each stay gives at most one review.
# A monthly+ listing with more reviews in the last 12 months than that cannot have been rented only for 30+ nights.
monthly = df[df["rental_model"] == "Monthly+ (30+)"]
implausible = monthly["number_of_reviews_ltm"] > 365 // monthly["minimum_nights"]
short = df[df["rental_model"] == "Short stay (1-6 nights)"]

pd.DataFrame({
    "monthly_listings": monthly.groupby("city_name").size(),
    "median_reviews_ltm_monthly": monthly.groupby("city_name")["number_of_reviews_ltm"].median(),
    "median_reviews_ltm_short_stay": short.groupby("city_name")["number_of_reviews_ltm"].median(),
    "implausible_pct": implausible.groupby(monthly["city_name"]).mean().mul(100).round(1),
})

## 9. Seasonal switching: short stays in summer, monthly rentals in winter?

The data is a snapshot from late June 2026, so `minimum_nights` shows the summer setting. A model often described in Barcelona is short stays in summer and monthly rentals for the rest of the year. If hosts plan this ahead in their calendar, it is visible in two columns that look 365 nights into the future: `minimum_minimum_nights` (the smallest minimum stay in the calendar) and `maximum_minimum_nights` (the largest). A listing **switches rental model** if its calendar contains both a short-stay minimum (≤ 6 nights) and a monthly+ minimum (≥ 30 nights).

**Team's prediction:** about 10% of Barcelona listings switch, and fewer in Los Angeles, where stays are longer anyway.

**Result:** the prediction was too high. Only **1.6% of Barcelona listings (246) and 1.5% of Los Angeles listings (654)** switch between a short-stay and a monthly+ minimum within the next year, so the two cities hardly differ. Changes of the minimum stay are common (42.7% of Barcelona listings and 20.9% of Los Angeles listings), but small: among listings whose minimum varies, the median largest minimum is only 4 nights, e.g. a longer minimum on weekends or in peak weeks. In Barcelona, no listing marked *exempt: seasonal rental* switches (0%), as expected: without a tourist licence a short stay is not legal. Switching is most common among listings with no licence entry (2.8%). In the calendar data, the rental model is therefore a stable choice of the host, and the summer snapshot is a fair picture of the rest of the year.

**Limitations:** the two columns show *that* the minimum changes within the next year, but not *when*. A host who changes the setting by hand in September, or moves the flat to another platform for the winter, is invisible in a single snapshot. Testing that needs an earlier (winter) snapshot of the same listings, matched by `id`.

In [ ]:
# A listing switches rental model if its calendar for the next 365 nights contains both
# a short-stay minimum (<= 6 nights) and a monthly+ minimum (>= 30 nights)
df["varies_min_nights"] = df["minimum_minimum_nights"] != df["maximum_minimum_nights"]
df["switches_model"] = (df["minimum_minimum_nights"] <= 6) & (df["maximum_minimum_nights"] >= 30)

pd.DataFrame({
    "varies_pct": df.groupby("city_name")["varies_min_nights"].mean().mul(100).round(1),
    "switches_pct": df.groupby("city_name")["switches_model"].mean().mul(100).round(1),
    "switches_n": df.groupby("city_name")["switches_model"].sum(),
    # how big is the change for listings whose minimum varies at all?
    "median_largest_min_if_varies": df[df["varies_min_nights"]].groupby("city_name")["maximum_minimum_nights"].median(),
})

In [ ]:
# Barcelona: share of listings that switch model, per licence status (licence_status comes from section 7)
switches_bcn = df.loc[bcn.index, "switches_model"]
pd.DataFrame({
    "listings": switches_bcn.groupby(licence_status).size(),
    "switches_pct": switches_bcn.groupby(licence_status).mean().mul(100).round(1),
    "switches_n": switches_bcn.groupby(licence_status).sum(),
})

## 10. Map: where are the monthly+ listings?

This is the geographic part of the study. The table gives the monthly+ share per district (neighbourhood group). Then, for each city, an interactive map saved in `maps/` as HTML shows the **share of monthly+ listings per neighbourhood** as a choropleth: the GeoJSON polygons are joined to the listings on `neighbourhood_cleansed`. The map uses a share and not a count, because a count would mostly show where listings are in general. Neighbourhoods with fewer than 20 listings are grey, since a share computed from a handful of listings is mostly noise. Two heat-map layers (short-stay and monthly+ listings) can be switched on to compare where each type is concentrated.

Interactive features: three base maps from different tile providers (TMS), a layer control, a custom sequential colour scale (one hue, light to dark), a tooltip on hover and a popup on click. The map functions live in `src/maps.py` and work for every city. Screenshots for the report: `figures/study04_map_monthly_share_<city>.png`.

**Team's prediction:** the monthly+ share is higher on the outskirts, while the tourist areas in the centre are short-stay.

**Result:**
- **Barcelona:** the monthly+ share ranges from 28.3% (Eixample) to 51.9% (Sarrià-Sant Gervasi) per district, and from 16.5% to 77.3% per neighbourhood (neighbourhoods with at least 20 listings). The prediction holds only in part. The outer districts Sarrià-Sant Gervasi, Sant Andreu and Horta-Guinardó do have high shares (47–52%). But **Ciutat Vella, the most touristic district, has the second-highest share (50.9%)**, while Eixample, also central, has the lowest (28.3%).
- **Los Angeles:** the share differs most between jurisdictions: **74.2% in the City of Los Angeles, 22.2% in the other cities of the county and 9.2% in the unincorporated areas**. On the map, the dark area follows the city boundary, including the narrow strip that connects the city to its port in San Pedro.

**Interpretation:** _to fill in by the team_

In [ ]:
def district_share(data: pd.DataFrame) -> pd.DataFrame:
    """Share of monthly+ listings (%) per district (neighbourhood group), from lowest to highest within each city."""
    table = data.groupby(["city_name", "neighbourhood_group_cleansed"])["is_monthly"].agg(listings="size", monthly_pct="mean")
    table["monthly_pct"] = table["monthly_pct"].mul(100).round(1)
    return table.sort_values(["city_name", "monthly_pct"])


df["is_monthly"] = df["rental_model"] == "Monthly+ (30+)"
district_share(df)

In [ ]:
def make_maps(data: pd.DataFrame, show: bool = True) -> None:
    """One interactive map per city: choropleth of the monthly+ share per neighbourhood, plus heat maps
    of both rental models as extra layers (off by default, switched on in the layer control)."""
    for city in data["city"].unique():
        city_df = data[data["city"] == city]
        by_nbhd = city_df.groupby("neighbourhood_cleansed")
        m = base_map(city_df)
        add_choropleth(m, city, by_nbhd["is_monthly"].mean() * 100, by_nbhd.size(),
                       name="Monthly+ listings", legend="Monthly+ listings (% of listings in the neighbourhood)")
        add_heatmap(m, city_df[city_df["rental_model"] == "Short stay (1-6 nights)"], name="Heat map: short-stay listings")
        add_heatmap(m, city_df[city_df["is_monthly"]], name="Heat map: monthly+ listings")
        path = save_map(m, f"study04_monthly_share_{city}.html")
        print("Saved", path.name)
        if show:
            display(m)


make_maps(df)

# Replication: pairs 2 and 3

The method is frozen after pair 1. The cells below load all six cities and apply **the same definitions and the same functions** as sections 1–10, with no threshold changed, so pairs 2 and 3 are a real test of the pair-1 findings. Pair 2 (Madrid – New York) reverses the regulation contrast of pair 1: in this pair the European city is the less strict one and the US city the stricter one (New York, Local Law 18). Pair 3 (Tokyo – Mexico City) tests whether the patterns hold outside Europe and the USA. Section 7 (licence field) is not replicated, because the field has a different format in every city.

**Expectation from the pair design (fixed before looking at the results):** in pair 1 the US city has the larger monthly+ share. If pair 2 repeats this (New York > Madrid) although the regulation contrast is reversed, the continent is a candidate driver. If it flips (Madrid > New York), regulation is. Pair 3 then shows which of the two explanations still works without a US or a European city.

## R1. Load all six cities

In [ ]:
# Load all six cities, ordered by pair, and apply the same definitions as in sections 4, 6, 8, 9 and 10
df_all = load_all(usecols=cols)
df_all["city_name"] = pd.Categorical(df_all["city"].map({k: c.name for k, c in CITIES.items()}),
                                     categories=[c.name for c in CITIES.values()], ordered=True)

df_all["is_inactive"] = (df_all["availability_365"] == 0) & (df_all["number_of_reviews_ltm"] == 0)
df_all["rental_model"] = pd.cut(df_all["minimum_nights"], bins=bins, labels=labels)  # same bins and labels as section 6
df_all["is_monthly"] = df_all["rental_model"] == "Monthly+ (30+)"
df_all["implausible"] = df_all["is_monthly"] & (df_all["number_of_reviews_ltm"] > 365 // df_all["minimum_nights"])
df_all["switches_model"] = (df_all["minimum_minimum_nights"] <= 6) & (df_all["maximum_minimum_nights"] >= 30)
df_all.groupby("city_name").size()

## R2. Key indicators per city

One row per city with the main numbers of sections 4–9: share of short-stay and monthly+ listings (section 6), the position of the second peak (section 5), share of inactive listings (section 4), share of listings that switch model (section 9) and the share of implausible monthly+ listings (section 8).

In [ ]:
# Key indicators of the study, one row per city (ordered by pair)
by_city = df_all.groupby("city_name")
monthly_rows = df_all[df_all["is_monthly"]]

summary = pd.DataFrame({
    "pair": by_city["city"].first().map(lambda key: CITIES[key].pair),
    "listings": by_city.size(),
    "short_stay_pct": by_city["rental_model"].apply(lambda s: (s == "Short stay (1-6 nights)").mean() * 100),
    "monthly_pct": by_city["is_monthly"].mean() * 100,
    # the most common minimum stay among monthly+ listings = where the second peak sits
    "monthly_peak_nights": monthly_rows.groupby("city_name")["minimum_nights"].agg(lambda s: s.mode().iloc[0]).astype(int),
    "inactive_pct": by_city["is_inactive"].mean() * 100,
    "switches_pct": by_city["switches_model"].mean() * 100,
    "implausible_pct_of_monthly": monthly_rows.groupby("city_name")["implausible"].mean() * 100,
}).round(1)
summary

## R3. Charts for all six cities

The two charts of sections 5 and 6, drawn by the same functions for all six cities: the distribution of `minimum_nights` (where is the second peak?) and the share of each rental model.

In [ ]:
plot_min_nights(df_all, "study04_min_nights_distribution_all.png")

In [ ]:
plot_rental_model_share(df_all, "study04_rental_model_share_all.png")

## R4. Geography: districts and maps

The district table and the maps of section 10, for all six cities. Tokyo and Mexico City have no neighbourhood groups, so they are missing from the district table, but they do have neighbourhood-level maps.

In [ ]:
district_share(df_all)

In [ ]:
# Same maps for all six cities (saved in maps/, not displayed here to keep the notebook light)
make_maps(df_all, show=False)

## R5. Replication result

Each pair-1 finding is checked on pair 2 and pair 3 with the same code. **Replicated** = holds in both cities of the pair, **Partial** = holds in one of them, **Not replicated** = holds in neither.

| Pair-1 finding (Barcelona, Los Angeles) | Pair 2: Madrid – New York | Pair 3: Tokyo – Mexico City |
|---|---|---|
| A large monthly+ segment: more than a third of listings require 30+ nights (37.1%, 46.8%) | **Partial:** New York 81.7%, Madrid 12.2% | **Not replicated:** Tokyo 12.0%, Mexico City 1.2% |
| The distribution has a second peak at about one month | **Partial:** strong in New York (78.7% of listings at exactly 30 nights), weak in Madrid (6.3% at 30, 3.6% at 31) | **Partial:** Tokyo yes (11.4% at 30), Mexico City no |
| The US city has a larger monthly+ share than the European city (46.8% vs 37.1%) | **Replicated:** 81.7% vs 12.2% | n/a (no US or European city) |
| Monthly+ listings are real monthly rentals (implausible: 0.5%, 1.1%) | **Replicated:** 1.2%, 0.1% | **Replicated:** 0.3%, 2.7% (of a very small group in Mexico City) |
| Switching between short stays and 30+ within a year is rare (1.6%, 1.5%) | **Partial:** New York 0.3%, Madrid 5.2% | **Replicated:** 1.8%, 0.5% |
| About 8% of listings are inactive (7.8%, 7.6%) | **Partial:** Madrid 10.2% is close, New York 21.3% is far higher | **Not replicated:** Tokyo 1.3%, Mexico City 2.8% |
| Team's prediction: the monthly+ share is higher on the outskirts (only partly true in Barcelona) | **Not replicated:** in Madrid the outer districts have the lowest shares (1–5%) and Chamberí (25.6%) and Salamanca (23.0%) the highest; in New York the share is high in every borough (78.6–85.2%) | n/a (no districts in the data; see the maps) |

**Interpretation (draft for the team to check):**
- **What the pair design says.** The pair-1 direction (US city > European city) repeats in pair 2, which on its own points to the continent. But two observations cannot be explained by the continent: pair 3, with neither a US nor a European city, shows the same kind of gap (Tokyo 12.0% vs Mexico City 1.2%, the stricter city again higher), and inside Los Angeles the share jumps at the city boundary (section 10). Regulation explains all three pairs and the Los Angeles boundary.
- **A sharper version of the hypothesis.** What matters is not how strict a city is in general, but whether its rules draw a hard line at about one month. Where such a line exists (Catalonia: 31 days; City of Los Angeles and New York: 30 days), the second peak sits exactly on it. The harder the rules make stays below the line, the larger the monthly+ segment: New York (short stays only with the host present, Local Law 18) > City of Los Angeles (only in the host's primary residence, max. 120 days) > Barcelona (frozen tourist licences, but about 7,400 still valid). Mexico City, the least strict city in the sample, has almost no monthly+ segment. Under this reading, the label "less strict" for Los Angeles in the pair design is too coarse: for a host without a primary-residence registration, the City of Los Angeles is very strict.
- **To verify before the report:** the Tokyo peak at 30 nights (Japan's Private Lodging Business Act of 2018 caps home-sharing at 180 nights a year, and stays of one month or more may fall under lease rules) and the Madrid rules.

**Open questions:** Why is Madrid so different from Barcelona, although both cities are under the same Spanish lease law? Why does New York have 21.3% inactive listings, and is this the same group as the 28.9% of New York listings without a price?

## Findings

### Results on pair 1 (discovery: Barcelona, Los Angeles)
- **Data quality:** `instant_bookable` is empty in all six cities. A missing `price` is not random: it marks listings with an empty calendar and no recent reviews, so these rows are kept. About 8% of listings in both cities are inactive ("ghost") listings.
- **Rental model:** the minimum stay is bimodal: 1–2 nights or about one month, almost nothing in between. Monthly+ listings (30+ nights) are **37.1% of Barcelona and 46.8% of Los Angeles**. The team's prediction (more in Los Angeles) holds.
- **The second peak sits on the legal line:** 31–32 nights in Barcelona (Catalan tourist rental = stays of 31 days or less), 30 nights in Los Angeles (Home-Sharing Ordinance = 30 days or less). In Barcelona, 77.3% of short-stay listings have a tourist licence against 1.4% of monthly+ listings, and 52.2% of monthly+ listings are marked *exempt: seasonal rental*.
- **Monthly+ is real and stable:** at most 0.5–1.1% of monthly+ listings got more reviews than a monthly rental allows, and only 1.5–1.6% of listings switch between short stays and 30+ within a year (the team's prediction for Barcelona was about 10%).
- **Geography:** in Los Angeles the monthly+ share is 74.2% inside the City of Los Angeles and 9–22% outside it, following the city boundary. In Barcelona it is high in both the tourist centre (Ciutat Vella, 50.9%) and the outer districts, and lowest in Eixample (28.3%).

### Cross-city comparison (replication on pairs 2 and 3)
- Monthly+ share: **New York 81.7%, Los Angeles 46.8%, Barcelona 37.1%, Madrid 12.2%, Tokyo 12.0%, Mexico City 1.2%.** Wherever a monthly peak exists, it sits at 30–32 nights.
- See the table in R5: the existence of a large monthly+ segment replicates only partly, while the checks on the quality of the monthly+ segment (real, stable) replicate.
- Main conclusion (draft): **the rental model on Airbnb follows local regulation, not the continent.** The monthly+ segment is the part of the market that hosts move to when the rules make short stays hard. Its size depends on how hard the line at about one month is enforced.

### Limitations
- **Snapshot:** June–July 2026 only. `minimum_nights` is a setting chosen by the host, not the length of actual stays. Seasonal changes made by hand are not visible.
- **No guest or booking data:** reviews are a proxy for stays, and the guests of monthly+ listings (students, remote workers) are an assumption.
- **One city per role:** six cities are not an experiment. Housing costs, tourist demand and Airbnb's own enforcement of registration numbers are possible confounders.
- The licence check works for Barcelona only, and part of the regulation facts come from secondary sources.
- Neighbourhoods differ in size between cities, and neighbourhoods with fewer than 20 listings are not coloured on the maps.